# Pyomo Tutorial - Example 7

**Original AMPL author:** Xingpeng Li (xli83@central.uh.edu, UH-ECE)
**Converted by:** Haoxiang Wan, PhD student of Dr. Xingpeng Li

Generic LP parametrized by sets $I$, $J$. Two datasets:

| | LP | Optimal |
|---|---|---|
| Ex-5 | $\max x_1{+}x_2$ s.t. $3x_1{+}4x_2\le 24,\;7x_1{+}4x_2\le 28$ | $x_1{=}1,\;x_2{=}5.25$ |
| Ex-6 | $\max 2x_1{+}5x_2{+}x_3$ s.t. $x_1{+}3x_2{+}2x_3\le 10,\;2x_1{+}x_2{+}5x_3\le 8$ | $x_2{=}10/3$ |

Cell 1 mirrors the original `.mod`'s 5 data-loading options 1:1.
**Default = option 1, Ex-5** (same as the AMPL default).


In [1]:
############# Load data - multiple options are available
import sys, pathlib
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data, parse_text

### option 1
#_d = parse_ampl_data('example_7_w_Ex5_Data_all_in_one.txt')    # for example 5
#_d = parse_ampl_data('example_7_w_Ex6_Data_all_in_one.txt')  # for example 6

### option 2  (in Python, identical to option 1)
# _d = parse_ampl_data('example_7_w_Ex5_Data_all_in_one.txt')  # for example 5
# _d = parse_ampl_data('example_7_w_Ex6_Data_all_in_one.txt')  # for example 6

### option 3
# _d = {**parse_ampl_data('example_7_w_Ex5_Data_a.txt'),
#       **parse_ampl_data('example_7_w_Ex5_Data_b.txt'),
#       **parse_ampl_data('example_7_w_Ex5_Data_c.txt')}

### option 4  (in Python, same data as option 3)
# _d = {}
# _d.update(parse_ampl_data('example_7_w_Ex5_Data_a.txt'))
# _d.update(parse_ampl_data('example_7_w_Ex5_Data_b.txt'))
# _d.update(parse_ampl_data('example_7_w_Ex5_Data_c.txt'))

### option 5
with open('example_7_w_Ex5_Data_b_noTitle.txt') as f: _b = f.read()
with open('example_7_w_Ex5_Data_c_noTitle.txt') as f: _c = f.read()
_d = parse_text(f'param: I: b := {_b}\nparam: J: c := {_c}')
_d.update(parse_ampl_data('example_7_w_Ex5_Data_a.txt'))

I_data, J_data = _d['I'], _d['J']
a_data, b_data, c_data = _d['a'], _d['b'], _d['c']
print(f'I={I_data}  J={J_data}')
print(f'a={a_data}')
print(f'b={b_data}  c={c_data}')


I=[1, 2]  J=[1, 2]
a={(1, 1): 3, (1, 2): 4, (2, 1): 7, (2, 2): 4}
b={1: 24, 2: 28}  c={1: 1, 2: 1}


In [2]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    NonNegativeReals, maximize, value
)

model = ConcreteModel()

model.I = Set(initialize=I_data)
model.J = Set(initialize=J_data)

model.a = Param(model.I, model.J, initialize=a_data)
model.b = Param(model.I, initialize=b_data)
model.c = Param(model.J, initialize=c_data)

model.x = Var(model.J, domain=NonNegativeReals)

model.obj = Objective(
    rule=lambda m: sum(m.c[j]*m.x[j] for j in m.J),
    sense=maximize
)
model.constName = Constraint(
    model.I,
    rule=lambda m, i: sum(m.a[i, j]*m.x[j] for j in m.J) <= m.b[i]
)

In [3]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)

for j in model.J:
    print(f"x[{j}] = {value(model.x[j])}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpbrqf_ohu.pyomo.lp
Reading time = 0.00 seconds
x1: 2 rows, 2 columns, 4 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90
MIPGap  0

Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0x2275f4a2
Coefficient statistics:
  Matrix range     [3e+00, 7e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 3e+01]
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+30   3.125000e+30   2.000000e+00      0s
       2    6.2500000e+00   0.000000e+00   0.000000e+00 

Reading time = 0.00 seconds
x1: 2 rows, 2 columns, 4 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90
MIPGap  0



Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0x2275f4a2
Coefficient statistics:
  Matrix range     [3e+00, 7e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 3e+01]


Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros



Iteration    Objective       Primal Inf.    Dual Inf.      Time


       0    2.0000000e+30   3.125000e+30   2.000000e+00      0s


       2    6.2500000e+00   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  6.250000000e+00


ok optimal
x[1] = 1.0
x[2] = 5.25
